---
title: "Project 01 · The NLP Time Machine"
subtitle: "One sentiment task, three eras of natural language processing"
author: "CS 351 · Introduction to Natural Language Processing"
format:
  html:
    toc: true
    toc-depth: 2
execute:
  enabled: false
---

In this two-week project, you will solve the same sentiment-classification problem with **hand-written rules**, **statistical learning**, and a **pretrained Transformer**. The notebook supplies data loading, evaluation, and plotting. You complete the small pieces that reveal what each era changed.

**Deliverable:** this completed notebook with every output preserved and a 250–400 word reflection.


## Before you begin

From the starter directory, install the locked environment and open this notebook:

```bash
uv sync
uv run jupyter lab nlp_time_machine.ipynb
```

Run cells in order. Each TODO is followed by an immediate check. A fresh notebook is expected to stop at the first `NotImplementedError`.


In [ ]:
from __future__ import annotations

import re
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

RANDOM_SEED = 351
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
LABEL_NAMES = np.array(["negative", "positive"])


## Load one task for all three eras

We use the Stanford Sentiment Treebank binary classification task (SST-2). To keep the project responsive, the statistical model trains on a deterministic sample of 6,000 examples and all systems share the same 500-example validation sample.


In [ ]:
dataset = load_dataset("stanfordnlp/sst2")
train_frame = dataset["train"].to_pandas().sample(
    n=6_000, random_state=RANDOM_SEED
).reset_index(drop=True)
test_frame = dataset["validation"].to_pandas().sample(
    n=500, random_state=RANDOM_SEED
).reset_index(drop=True)

train_frame["label_name"] = LABEL_NAMES[train_frame["label"].to_numpy()]
test_frame["label_name"] = LABEL_NAMES[test_frame["label"].to_numpy()]
test_frame[["sentence", "label_name"]].head()


# Era I · Tell the machine the rules

A rule-based classifier can only use knowledge we explicitly provide. You will implement its tokenizer, sentiment score, and final decision.


In [ ]:
POSITIVE_WORDS = {
    "amazing", "best", "clear", "excellent", "fast", "good",
    "helpful", "impressive", "love", "reliable", "smooth", "wonderful",
}
NEGATIVE_WORDS = {
    "awful", "bad", "broken", "confusing", "disappointing", "hate",
    "poor", "slow", "terrible", "unreliable", "useless", "worst",
}
NEGATIONS = {"no", "not", "never"}


## TODO 1 · Tokenize the review

Return lowercase alphabetic tokens while preserving internal apostrophes. For example, `It's NOT reliable!` becomes `["it's", "not", "reliable"]`.


In [ ]:
def tokenize(text: str) -> list[str]:
    # TODO: use re.findall() on a lowercased version of text.
    raise NotImplementedError("Complete TODO 1")


In [ ]:
assert tokenize("Amazing, CLEAR results!") == ["amazing", "clear", "results"]
assert tokenize("It's helpful; don't stop.") == ["it's", "helpful", "don't", "stop"]
print("TODO 1 passed")


## TODO 2 · Score sentiment words

Positive words add `1`, negative words subtract `1`, and an immediately preceding negation reverses a sentiment contribution.


In [ ]:
def lexicon_score(tokens: Sequence[str]) -> int:
    # TODO: accumulate sentiment contributions and reverse after a negation.
    raise NotImplementedError("Complete TODO 2")


In [ ]:
assert lexicon_score(["good", "and", "reliable"]) == 2
assert lexicon_score(["slow", "and", "confusing"]) == -2
assert lexicon_score(["not", "good"]) == -1
assert lexicon_score(["never", "bad"]) == 1
print("TODO 2 passed")


## TODO 3 · Turn the score into a prediction

Compose the two functions above. Return `positive`, `negative`, or `neutral`.


In [ ]:
def rule_based_predict(text: str) -> str:
    # TODO: tokenize, score, and map the score sign to a label.
    raise NotImplementedError("Complete TODO 3")


In [ ]:
assert rule_based_predict("An excellent and reliable tool") == "positive"
assert rule_based_predict("A terrible, confusing tool") == "negative"
assert rule_based_predict("The tool arrived yesterday") == "neutral"
print("TODO 3 passed")


## Evaluate Era I

SST-2 has only positive and negative labels. A neutral rule prediction therefore counts as an error: it reveals that the lexicon did not cover enough of the review.


In [ ]:
rule_predictions = test_frame["sentence"].map(rule_based_predict).to_numpy()
rule_accuracy = accuracy_score(test_frame["label_name"], rule_predictions)
print(f"Rule accuracy: {rule_accuracy:.1%}")
test_frame.assign(rule=rule_predictions).query("rule == 'neutral'").head(5)


# Era II · Learn weights from examples

A statistical system learns which words and short phrases predict sentiment. You will fit TF–IDF only on training text, train logistic regression, and inspect its strongest learned features.


## TODO 4 · Fit the statistical classifier

Use unigram and bigram TF–IDF features. Fit the vectorizer only on training text, then fit logistic regression on the resulting matrix.


In [ ]:
def fit_statistical_classifier(
    texts: Sequence[str], labels: Sequence[str]
) -> tuple[TfidfVectorizer, LogisticRegression]:
    # TODO: create TfidfVectorizer(ngram_range=(1, 2), min_df=2).
    # TODO: call fit_transform(texts).
    # TODO: fit LogisticRegression(max_iter=1_000, random_state=RANDOM_SEED).
    raise NotImplementedError("Complete TODO 4")


In [ ]:
vectorizer, statistical_model = fit_statistical_classifier(
    train_frame["sentence"], train_frame["label_name"]
)
assert vectorizer.ngram_range == (1, 2)
assert len(statistical_model.classes_) == 2
print("TODO 4 passed")


## TODO 5 · Predict without leaking test data

Transform the new text with the already-fitted vectorizer, then call the classifier's prediction method. Do not fit anything in this function.


In [ ]:
def statistical_predict(
    texts: Sequence[str],
    vectorizer: TfidfVectorizer,
    model: LogisticRegression,
) -> np.ndarray:
    # TODO: transform texts and return model predictions.
    raise NotImplementedError("Complete TODO 5")


In [ ]:
statistical_predictions = statistical_predict(
    test_frame["sentence"], vectorizer, statistical_model
)
assert statistical_predictions.shape == (len(test_frame),)
assert set(statistical_predictions) <= {"negative", "positive"}
statistical_accuracy = accuracy_score(
    test_frame["label_name"], statistical_predictions
)
print(f"Statistical accuracy: {statistical_accuracy:.1%}")


## TODO 6 · Inspect what the model learned

Return the `n` features with the most negative and most positive logistic-regression weights.


In [ ]:
def strongest_features(
    vectorizer: TfidfVectorizer, model: LogisticRegression, n: int = 10
) -> tuple[list[str], list[str]]:
    # TODO: get feature names and sort model.coef_[0].
    # Return (most_negative, most_positive).
    raise NotImplementedError("Complete TODO 6")


In [ ]:
negative_features, positive_features = strongest_features(
    vectorizer, statistical_model
)
assert len(negative_features) == len(positive_features) == 10
pd.DataFrame({"most negative": negative_features, "most positive": positive_features})


# Era III · Reuse pretrained knowledge

The final system was pretrained on far more text than our 6,000 examples and fine-tuned for SST-2 sentiment. You will still touch every stage of inference: load its paired tokenizer and model, encode a batch, compute logits, and decode labels.


## TODO 7 · Load the tokenizer and model

Both objects must use the same `MODEL_NAME`. The first run downloads and caches them.


In [ ]:
def load_transformer():
    # TODO: load AutoTokenizer and AutoModelForSequenceClassification.
    raise NotImplementedError("Complete TODO 7")


In [ ]:
transformer_tokenizer, transformer_model = load_transformer()
assert transformer_tokenizer.name_or_path == MODEL_NAME
assert transformer_model.config.num_labels == 2
print("TODO 7 passed")


## TODO 8 · Encode a batch

Use padding, truncation, and PyTorch tensors. A maximum length of 128 is sufficient for SST-2.


In [ ]:
def encode_batch(texts: Sequence[str], tokenizer):
    # TODO: call tokenizer with padding, truncation, max_length=128,
    # and return_tensors="pt".
    raise NotImplementedError("Complete TODO 8")


In [ ]:
sample_batch = encode_batch(
    ["A wonderful film.", "A disappointing film."], transformer_tokenizer
)
assert sample_batch["input_ids"].shape[0] == 2
assert sample_batch["attention_mask"].shape == sample_batch["input_ids"].shape
transformer_tokenizer.convert_ids_to_tokens(sample_batch["input_ids"][0])


## TODO 9 · Compute predictions

Process texts in batches. Inside `torch.inference_mode()`, obtain logits, select the largest logit with `argmax`, and map IDs through `model.config.id2label`. Return lowercase labels.


In [ ]:
def transformer_predict(
    texts: Sequence[str], tokenizer, model, batch_size: int = 32
) -> np.ndarray:
    # TODO: loop over batches, call encode_batch(), run the model,
    # select argmax labels, and return one lowercase NumPy array.
    raise NotImplementedError("Complete TODO 9")


In [ ]:
demo_predictions = transformer_predict(
    ["A wonderful film.", "A disappointing film."],
    transformer_tokenizer,
    transformer_model,
)
assert demo_predictions.tolist() == ["positive", "negative"]
print("TODO 9 passed")


# Compare the three eras

You have now implemented part of every system. Run the same 500 reviews through all three and make the historical progression visible.


In [ ]:
transformer_predictions = transformer_predict(
    test_frame["sentence"].tolist(),
    transformer_tokenizer,
    transformer_model,
)
transformer_accuracy = accuracy_score(
    test_frame["label_name"], transformer_predictions
)

results = pd.DataFrame(
    {
        "era": ["Rules", "TF–IDF + logistic regression", "Pretrained Transformer"],
        "accuracy": [rule_accuracy, statistical_accuracy, transformer_accuracy],
    }
).sort_values("accuracy")
results.style.format({"accuracy": "{:.1%}"})


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(results["era"], results["accuracy"], color=["#b48a32", "#0f6f78", "#315f7d"])
ax.bar_label(bars, labels=[f"{value:.1%}" for value in results["accuracy"]], padding=4)
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy on the same 500 SST-2 reviews")
ax.set_title("Three eras of sentiment classification")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## Compare error patterns

Accuracy summarizes performance; disagreements explain it. The next table prioritizes examples where a later era corrects an earlier one.


In [ ]:
comparison = test_frame[["sentence", "label_name"]].copy()
comparison["rules"] = rule_predictions
comparison["statistical"] = statistical_predictions
comparison["transformer"] = transformer_predictions
comparison["rules_correct"] = comparison["rules"] == comparison["label_name"]
comparison["statistical_correct"] = comparison["statistical"] == comparison["label_name"]
comparison["transformer_correct"] = comparison["transformer"] == comparison["label_name"]

comparison.query(
    "not rules_correct and not statistical_correct and transformer_correct"
)[["sentence", "label_name", "rules", "statistical", "transformer"]].head(10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
prediction_columns = ["rules", "statistical", "transformer"]
titles = ["Rules", "Statistical", "Transformer"]
matrix_labels = ["negative", "neutral", "positive"]

for ax, column, title in zip(axes, prediction_columns, titles):
    matrix = confusion_matrix(
        comparison["label_name"], comparison[column],
        labels=matrix_labels,
    )
    ConfusionMatrixDisplay(
        matrix, display_labels=matrix_labels
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)

plt.tight_layout()
plt.show()


# Final reflection · 250–400 words

Replace this paragraph with your response:

> What knowledge did each era require from humans, labeled examples, and pretraining? Use at least three specific disagreements from your results. Explain one capability gained in each transition and one kind of error that still remains in the pretrained Transformer.


## Submission checklist

- [ ] TODOs 1–9 are complete and every check passes.
- [ ] The notebook runs from top to bottom after restarting the kernel.
- [ ] The accuracy table, bar chart, confusion matrices, and disagreement table are visible.
- [ ] The final reflection uses evidence from your own results.
- [ ] All output cells are preserved when you submit the notebook.
